# Multimodal Generative AI: VLMs, Contrastive Learning, Diffusion & Multimodal RAG

This notebook implements the architectural pillars of modern multimodal AI:
1. **Contrastive Vision-Language Learning (CLIP)**: Joint dual encoders, symmetric InfoNCE loss, and zero-shot transfer.
2. **Vision-Language Model (VLM) Projectors (LLaVA style)**: Mapping visual patch tokens into an autoregressive LLM's sequence space.
3. **Audio-Language Projection**: Downsampling continuous temporal spectrogram frames into discrete LLM tokens.
4. **Diffusion & Classifier-Free Guidance (CFG)**: Forward noise scheduling and conditional guidance dynamics.
5. **Multimodal RAG**: Indexing heterogeneous text passages and image embeddings for cross-modal search.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/multimodal/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import multimodal_core
print('multimodal_core loaded')

## 1. Contrastive Vision-Language Pretraining (CLIP)
Dual encoders project images and text into a shared hypersphere where positive pairs attract and negative pairs repel.

In [ ]:
import torch
import torch.nn.functional as F
from multimodal_core import CLIPModel, clip_loss, zero_shot_classifier

torch.manual_seed(42)
B = 4
images = torch.randn(B, 3, 16, 16)
texts = torch.randint(0, 100, (B, 8))

clip = CLIPModel(d_vision=32, d_text=32, embed_dim=16)
logits_img, logits_txt = clip(images, texts)

loss = clip_loss(logits_img, logits_txt)
print(f"CLIP Symmetric InfoNCE Loss: {loss.item():.4f}")
print("Image-to-Text Logits shape  :", logits_img.shape)

# Zero-shot classification test
candidate_prompts = torch.randint(0, 100, (3, 8))  # 3 candidate class descriptions
probs = zero_shot_classifier(clip, images[:2], candidate_prompts)
print("Zero-shot probabilities for 2 test images across 3 candidate classes:")
print(probs)

## 2. Vision-Language Model (VLM) Projector (LLaVA Pattern)
Rather than fine-tuning the LLM backbone, a lightweight MLP projects vision transformer patch features into the LLM token embedding dimension.

In [ ]:
from multimodal_core import VLMProjector, build_multimodal_input

num_patches = 16
d_vision = 32
d_llm = 64

patch_tokens = torch.randn(2, num_patches, d_vision)
vlm_proj = VLMProjector(d_vision=d_vision, d_llm=d_llm)

# Project image patches into text embedding space
image_tokens = vlm_proj(patch_tokens)
print(f"Projected visual tokens: {image_tokens.shape} -> (Batch, Num_Patches, D_LLM)")

# Text prompt token embeddings
text_embeddings = torch.randn(2, 12, d_llm)

# Prefix injection: [Image Tokens || Text Tokens]
full_sequence = build_multimodal_input(image_tokens, text_embeddings)
print(f"Unified sequence length: {full_sequence.shape[1]} tokens (16 image + 12 text)")

## 3. Audio-Language Frame Projection
Mel-spectrograms pass through 1D strided convolution to compress time before entering the LLM embedding stream.

In [ ]:
from multimodal_core import AudioFrameProjector

n_mels = 80
time_steps = 60
d_llm = 64

mel_input = torch.randn(2, n_mels, time_steps)
aud_proj = AudioFrameProjector(n_mels=n_mels, d_llm=d_llm, stride=2)

audio_tokens = aud_proj(mel_input)
print(f"Compressed audio tokens: {audio_tokens.shape} (Time downsampled {time_steps} -> {audio_tokens.shape[1]})")

## 4. Diffusion Forward Noise Scheduling & Classifier-Free Guidance (CFG)
Forward diffusion adds Gaussian noise according to schedule $\bar{\alpha}_t$; CFG magnifies prompt conditioning.

In [ ]:
from multimodal_core import DiffusionNoiseSchedule, classifier_free_guidance

schedule = DiffusionNoiseSchedule(num_timesteps=1000)
x0 = torch.randn(2, 3, 32, 32)
timesteps = torch.tensor([50, 500])

# Diffuse image to step t
xt = schedule.q_sample(x0, timesteps)
print(f"Diffused sample xt shape: {xt.shape}")
print(f"Alpha_bar at step 50 : {schedule.alphas_cumprod[50].item():.4f}")
print(f"Alpha_bar at step 500: {schedule.alphas_cumprod[500].item():.4f}")

# Classifier-Free Guidance extrapolation
eps_unconditional = torch.tensor([0.2, 0.4])
eps_conditional = torch.tensor([0.6, 0.8])
eps_guided = classifier_free_guidance(eps_unconditional, eps_conditional, guidance_scale=7.5)
print(f"CFG Extrapolated noise (scale=7.5): {eps_guided.tolist()}")

## 5. Multimodal RAG Store & Cross-Modal Retrieval
A shared embedding index serving both text documents and images.

In [ ]:
from multimodal_core import MultimodalStore

store = MultimodalStore()

# Embeddings in shared 4D space
emb_text_diag = torch.tensor([0.9, 0.1, 0.0, 0.0])
emb_img_flowchart = torch.tensor([0.85, 0.15, 0.0, 0.0])
emb_text_finance = torch.tensor([0.0, 0.0, 0.9, 0.2])

store.add("doc_1", "text", "System architecture diagram explanation", emb_text_diag)
store.add("img_1", "image", "assets/architecture_flowchart.png", emb_img_flowchart)
store.add("doc_2", "text", "Quarterly financial earnings report", emb_text_finance)

# Query with text about system architecture
query = torch.tensor([0.95, 0.05, 0.0, 0.0])
results = store.search(query, top_k=2)

print("Top retrieved cross-modal items:")
for item, score in results:
    print(f" - [{item.modality.upper()}] id={item.id} (Score={score:.4f}): {item.content}")